# Driver Ranking Analysis


In [ ]:
%pip install pandas
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip in

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load and Prepare Data

In [11]:
# Load the lap-weather dataset from Luis
df = pd.read_pickle('../data/f1_lap_weather_data.pkl')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

Data shape: (188482, 41)

Columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'TrackStatus', 'Position', 'FastF1Generated', 'IsAccurate', 'Year', 'Location', 'EventName', 'LapStartTimeUTC', 'IsPitLap', 'IsTerminalLap', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 'WindSpeed', 'Rainfall']

Data types:
Time                  timedelta64[ns]
Driver                         object
DriverNumber                    int64
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time 

In [12]:
# Check data quality
print(f"Unique races: {df['EventName'].nunique()}")
print(f"Unique drivers: {df['Driver'].nunique()}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nPosition column null count: {df['Position'].isna().sum()} / {len(df)}")
print(f"Position data type: {df['Position'].dtype}")
print(df[['Driver', 'EventName', 'Year', 'LapNumber', 'Position', 'Team']].head(10))

Unique races: 36
Unique drivers: 43
Year range: 2018 - 2025

Position column null count: 2 / 188482
Position data type: float64
  Driver              EventName  Year  LapNumber  Position          Team
0    GAS  Australian Grand Prix  2018        1.0      17.0  Racing Bulls
1    GAS  Australian Grand Prix  2018        2.0      17.0  Racing Bulls
2    GAS  Australian Grand Prix  2018        3.0      17.0  Racing Bulls
3    GAS  Australian Grand Prix  2018        4.0      17.0  Racing Bulls
4    GAS  Australian Grand Prix  2018        5.0      17.0  Racing Bulls
5    GAS  Australian Grand Prix  2018        6.0      16.0  Racing Bulls
6    GAS  Australian Grand Prix  2018        7.0      16.0  Racing Bulls
7    GAS  Australian Grand Prix  2018        8.0      16.0  Racing Bulls
8    GAS  Australian Grand Prix  2018        9.0      16.0  Racing Bulls
9    GAS  Australian Grand Prix  2018       10.0      16.0  Racing Bulls


In [13]:
# cast Position as numeric
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# unique race identifier
df['RaceID'] = df['EventName'] + ' ' + df['Year'].astype(str)

print(f"Races with data: {df['RaceID'].nunique()}")
print(f"\nSample races:")
print(df['RaceID'].unique()[:5])

Races with data: 172

Sample races:
['Australian Grand Prix 2018' 'Bahrain Grand Prix 2018'
 'Chinese Grand Prix 2018' 'Azerbaijan Grand Prix 2018'
 'Spanish Grand Prix 2018']


## Create Race-Level Summary

In [14]:
race_results_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    
    # Get unique drivers in this race
    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()
    
    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()
        
        if len(driver_race) == 0:
            continue
            
        # Extract race info
        event_name = race_data['EventName'].iloc[0]
        year = race_data['Year'].iloc[0]
        location = race_data['Location'].iloc[0]
        
        # Position tracking
        valid_positions = driver_race['Position'].dropna()
        
        if len(valid_positions) == 0:
            continue
        
        # Grid position (first lap position)
        first_lap = driver_race[driver_race['LapNumber'] == 1.0]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan
        
        # Finishing position (last valid position)
        finishing_pos = valid_positions.iloc[-1]
        
        # Position statistics
        best_pos = valid_positions.min()
        worst_pos = valid_positions.max()
        avg_pos = valid_positions.mean()
        pos_std = valid_positions.std()
        
        # Positions gained/lost 
        if pd.notna(grid_pos):
            positions_gained = grid_pos - finishing_pos
        else:
            positions_gained = np.nan
        
        # Laps led
        laps_led = len(driver_race[driver_race['Position'] == 1.0])
        
        # Pit stops
        pit_stops = len(driver_race[driver_race['IsPitLap'] == True])
        
        # DNF detection 
        max_laps_any_driver = race_data['LapNumber'].max()
        driver_max_laps = driver_race['LapNumber'].max()
        dnf = driver_max_laps < (max_laps_any_driver - 2)  # Allow 2 lap tolerance
        
        race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'EventName': event_name,
            'Year': year,
            'Location': location,
            'GridPosition': grid_pos,
            'FinishingPosition': finishing_pos,
            'BestPosition': best_pos,
            'WorstPosition': worst_pos,
            'AvgPosition': avg_pos,
            'PositionStdDev': pos_std,
            'PositionsGained': positions_gained,
            'LapsLed': laps_led,
            'PitStops': pit_stops,
            'LapsCompleted': int(driver_max_laps),
            'DNF': dnf
        })

race_results_df = pd.DataFrame(race_results_list)
print(f"race_results_df: {race_results_df.shape}")
print(f"\nFirst race:")
print(race_results_df.head(19))

race_results_df: (3421, 17)

First race:
   Driver  DriverNumber             Team              EventName  Year  \
0     GAS            10     Racing Bulls  Australian Grand Prix  2018   
1     PER            11     Aston Martin  Australian Grand Prix  2018   
2     ALO            14          McLaren  Australian Grand Prix  2018   
3     LEC            16      Kick Sauber  Australian Grand Prix  2018   
4     STR            18         Williams  Australian Grand Prix  2018   
5     VAN             2          McLaren  Australian Grand Prix  2018   
6     MAG            20             Haas  Australian Grand Prix  2018   
7     HUL            27           Alpine  Australian Grand Prix  2018   
8     HAR            28     Racing Bulls  Australian Grand Prix  2018   
9     RIC             3  Red Bull Racing  Australian Grand Prix  2018   
10    OCO            31     Aston Martin  Australian Grand Prix  2018   
11    VER            33  Red Bull Racing  Australian Grand Prix  2018   
12    SIR 

In [15]:
print("race_results_df summary:")
print(race_results_df.describe())
print(f"\nDNF count: {race_results_df['DNF'].sum()}")
print(f"Races with drivers leading: {race_results_df[race_results_df['LapsLed'] > 0].shape[0]}")

race_results_df summary:
       DriverNumber         Year  GridPosition  FinishingPosition  \
count   3421.000000  3421.000000   3421.000000        3421.000000   
mean      27.888921  2021.661795     10.436714          10.235896   
std       24.471309     2.306975      5.724053           5.615003   
min        1.000000  2018.000000      1.000000           1.000000   
25%       10.000000  2020.000000      5.000000           5.000000   
50%       20.000000  2022.000000     10.000000          10.000000   
75%       44.000000  2024.000000     15.000000          15.000000   
max       99.000000  2025.000000     20.000000          20.000000   

       BestPosition  WorstPosition  AvgPosition  PositionStdDev  \
count   3421.000000    3421.000000  3421.000000     3340.000000   
mean       7.346098      13.723473    10.133731        1.743775   
std        4.791324       5.513842     5.192018        1.050418   
min        1.000000       1.000000     1.000000        0.000000   
25%        3.00000

## Create Driver-Level Statistics 

In [16]:
#statistics per driver across all races participated in
driver_stats_list = []

for driver in race_results_df['Driver'].unique():
    driver_races = race_results_df[race_results_df['Driver'] == driver]
    
    # driver info
    driver_num = driver_races['DriverNumber'].iloc[0]
    team = driver_races['Team'].iloc[0]
    
    # race counts
    races_entered = len(driver_races)
    races_completed = len(driver_races[driver_races['DNF'] == False])
    dnf_count = races_entered - races_completed
    
    # Position statistics
    avg_grid_pos = driver_races['GridPosition'].mean()
    avg_finish_pos = driver_races['FinishingPosition'].mean()
    
    # Position gained/lost
    avg_positions_gained = driver_races['PositionsGained'].mean()
    races_with_gains = len(driver_races[driver_races['PositionsGained'] > 0])
    races_with_losses = len(driver_races[driver_races['PositionsGained'] < 0])
    
    # Driver Performance metrics
    podiums = len(driver_races[driver_races['FinishingPosition'] <= 3])
    poles = len(driver_races[driver_races['GridPosition'] == 1.0])
    wins = len(driver_races[driver_races['FinishingPosition'] == 1.0])
    total_laps_led = driver_races['LapsLed'].sum()
    
    # DriverConsistency
    avg_position_std = driver_races['PositionStdDev'].mean()
    
    driver_stats_list.append({
        'Driver': driver,
        'DriverNumber': driver_num,
        'Team': team,
        'RacesEntered': races_entered,
        'RacesCompleted': races_completed,
        'DNFCount': dnf_count,
        'DNFRate': dnf_count / races_entered if races_entered > 0 else 0,
        'AvgGridPosition': avg_grid_pos,
        'AvgFinishPosition': avg_finish_pos,
        'AvgPositionsGained': avg_positions_gained,
        'RacesWithGains': races_with_gains,
        'RacesWithLosses': races_with_losses,
        'Podiums': podiums,
        'Poles': poles,
        'Wins': wins,
        'TotalLapsLed': total_laps_led,
        'AvgPositionConsistency': avg_position_std
    })

driver_stats_df = pd.DataFrame(driver_stats_list).sort_values('RacesEntered', ascending=False)
print(f"Driver Statistics DataFrame shape: {driver_stats_df.shape}")
print(f"\nTop 10 drivers (2018-2025):")
print(driver_stats_df.head(10))

Driver Statistics DataFrame shape: (43, 17)

Top 10 drivers (2018-2025):
   Driver  DriverNumber             Team  RacesEntered  RacesCompleted  \
11    VER            33  Red Bull Racing           172             154   
0     GAS            10     Racing Bulls           171             148   
13    HAM            44         Mercedes           171             162   
3     LEC            16      Kick Sauber           170             147   
4     STR            18         Williams           168             143   
15    SAI            55           Alpine           168             147   
22    NOR             4          McLaren           152             139   
23    RUS            63         Williams           152             135   
10    OCO            31     Aston Martin           150             127   
17    BOT            77         Mercedes           148             129   

    DNFCount   DNFRate  AvgGridPosition  AvgFinishPosition  \
11        18  0.104651         4.139535           

In [17]:
# Show drivers with most position gains
print("\nTop 10 drivers by average positions gained per race:")
print(driver_stats_df.nlargest(10, 'AvgPositionsGained')[['Driver', 'Team', 'AvgPositionsGained', 'RacesEntered']])

print("\nMost successful drivers (by wins):")
print(driver_stats_df.nlargest(10, 'Wins')[['Driver', 'Team', 'Wins', 'Podiums', 'Poles']])


Top 10 drivers by average positions gained per race:
   Driver          Team  AvgPositionsGained  RacesEntered
5     VAN       McLaren            3.050000            20
21    KVY  Racing Bulls            2.052632            38
37    BEA       Ferrari            1.407407            27
8     HAR  Racing Bulls            1.400000            20
19    ERI   Kick Sauber            1.400000            20
32    ZHO   Kick Sauber            1.352941            68
33    DEV      Williams            1.272727            11
26    LAT      Williams            1.229508            61
1     PER  Aston Martin            1.191781           146
16    RAI       Ferrari            0.769231            78

Most successful drivers (by wins):
   Driver             Team  Wins  Podiums  Poles
11    VER  Red Bull Racing    68      116     57
13    HAM         Mercedes    41       87     29
22    NOR          McLaren    11       46     10
35    PIA          McLaren     9       26      7
3     LEC      Kick Sauber 

## Create Lap-by-Lap Position Tracking

In [18]:
# Create a pivot table for lap-by-lap position tracking

position_tracking_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    
    race_pivot = race_data.pivot_table(
        index=['Driver', 'Team', 'DriverNumber'],
        columns='LapNumber',
        values='Position',
        aggfunc='first'
    )
    
    # race identifiers
    race_pivot['EventName'] = event_name
    race_pivot['Year'] = year
    race_pivot['RaceID'] = race_id
    
    position_tracking_list.append(race_pivot)

print(f"\nFirst race (first 5 drivers):")
print(position_tracking_list[0].head(5))


First race (first 5 drivers):
LapNumber                          1.0   2.0   3.0   4.0   5.0   6.0   7.0  \
Driver Team         DriverNumber                                             
ALO    McLaren      14            10.0  10.0  10.0  10.0  10.0  10.0  10.0   
BOT    Mercedes     77            15.0  15.0  15.0  14.0  14.0  14.0  14.0   
ERI    Kick Sauber  9             16.0  16.0  16.0  16.0  16.0  18.0   NaN   
GAS    Racing Bulls 10            17.0  17.0  17.0  17.0  17.0  16.0  16.0   
GRO    Haas         8              6.0   6.0   6.0   6.0   6.0   6.0   6.0   

LapNumber                          8.0   9.0  10.0  ...  52.0  53.0  54.0  \
Driver Team         DriverNumber                    ...                     
ALO    McLaren      14            10.0  10.0  10.0  ...   5.0   5.0   5.0   
BOT    Mercedes     77            14.0  13.0  13.0  ...   8.0   8.0   8.0   
ERI    Kick Sauber  9              NaN   NaN   NaN  ...   NaN   NaN   NaN   
GAS    Racing Bulls 10            16.

## Position Change Analysis

In [19]:
# dataframe tracking position changes within races
position_changes_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    location = race_data['Location'].iloc[0]
    
    for driver in race_data['Driver'].unique():
        driver_data = race_data[race_data['Driver'] == driver].sort_values('LapNumber')
        
        # Remove rows with NaN 
        driver_data = driver_data[driver_data['Position'].notna()]
        
        if len(driver_data) < 2:
            continue
        
        # Get driver info
        team = driver_data['Team'].iloc[0]
        driver_num = driver_data['DriverNumber'].iloc[0]
        
        # Track position changes lap by lap
        positions = driver_data['Position'].values
        lap_numbers = driver_data['LapNumber'].values
        
        # Calculate position changes
        for i in range(1, len(positions)):
            position_change = positions[i-1] - positions[i]  
            
            position_changes_list.append({
                'Driver': driver,
                'DriverNumber': driver_num,
                'Team': team,
                'EventName': event_name,
                'Year': year,
                'Location': location,
                'RaceID': race_id,
                'FromLap': int(lap_numbers[i-1]),
                'ToLap': int(lap_numbers[i]),
                'PositionBefore': positions[i-1],
                'PositionAfter': positions[i],
                'PositionChange': position_change
            })

position_changes_df = pd.DataFrame(position_changes_list)
print(f"position_changes_df: {position_changes_df.shape}")
print(f"\nFirst 10 position changes:")
print(position_changes_df.head(10))

position_changes_df: (185059, 12)

First 10 position changes:
  Driver  DriverNumber          Team              EventName  Year   Location  \
0    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
1    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
2    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
3    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
4    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
5    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
6    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
7    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
8    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
9    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   

                       RaceID  FromLap  ToLap  PositionBe

In [20]:
print("\nMost dramatic position gains in single lap:")
print(position_changes_df.nlargest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print("\nMost dramatic position losses in single lap:")
print(position_changes_df.nsmallest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print(f"\nAverage position changes per lap: {position_changes_df['PositionChange'].mean():.3f}")
print(f"Median position changes per lap: {position_changes_df['PositionChange'].median():.3f}")


Most dramatic position gains in single lap:
       Driver             EventName  Year  FromLap  PositionChange
70687     STR    Styrian Grand Prix  2021        4            13.0
122756    ZHO      Dutch Grand Prix  2023        2            13.0
73812     HAM  Hungarian Grand Prix  2021        3            12.0
122205    GAS      Dutch Grand Prix  2023        2            12.0
47131     MAG  Hungarian Grand Prix  2020        3            11.0
122418    LEC      Dutch Grand Prix  2023        2            11.0
45020     LAT   Austrian Grand Prix  2020       48            10.0
122276    PER      Dutch Grand Prix  2023        2            10.0
45038     LAT   Austrian Grand Prix  2020       66             9.0
47539     GRO  Hungarian Grand Prix  2020        3             9.0

Most dramatic position losses in single lap:
       Driver              EventName  Year  FromLap  PositionChange
97484     LEC      French Grand Prix  2022       17           -19.0
9502      BOT    Austrian Grand Prix

## Exporting Dataframes

In [21]:
# Save the dataframes to CSV 
race_results_df.to_csv('../data/race_results_summary.csv', index=False)
driver_stats_df.to_csv('../data/driver_statistics.csv', index=False)
position_changes_df.to_csv('../data/position_changes.csv', index=False)


print(f"  - race_results_df: {race_results_df.shape}")
print(f"  - driver_stats_df: {driver_stats_df.shape}")
print(f"  - position_changes_df: {position_changes_df.shape}")

  - race_results_df: (3421, 17)
  - driver_stats_df: (43, 17)
  - position_changes_df: (185059, 12)


---
## Rainy Condition Analysis

The following sections replicate the race-level summary, driver statistics, and position change analysis from above, restricted to laps where **`Rainfall == True`**. Clean-lap filters are also applied (TrackStatus == 1, no pit laps, no terminal laps) to isolate meaningful racing laps in wet conditions.

### Filter to Rainy Clean Laps

In [22]:
# Filter to rainy, clean racing laps only
df_rain = df[
    (df['Rainfall'] == True)
    & (df['TrackStatus'] == 1)
    & (df['IsPitLap'] == False)
    & (df['IsTerminalLap'] == False)
].copy()

df_rain['RaceID'] = df_rain['EventName'] + ' ' + df_rain['Year'].astype(str)

print(f"Total laps (original): {len(df)}")
print(f"Rainy clean laps: {len(df_rain)}")
print(f"Rainy races: {df_rain['RaceID'].nunique()}")
print(f"\nRainy races included:")
print(df_rain['RaceID'].unique())

Total laps (original): 188482
Rainy clean laps: 5525
Rainy races: 22

Rainy races included:
['Spanish Grand Prix 2018' 'Monaco Grand Prix 2018'
 'German Grand Prix 2018' 'Abu Dhabi Grand Prix 2018'
 'Monaco Grand Prix 2019' 'German Grand Prix 2019'
 'Emilia Romagna Grand Prix 2021' 'Belgian Grand Prix 2021'
 'Russian Grand Prix 2021' 'Monaco Grand Prix 2022'
 'Hungarian Grand Prix 2022' 'Japanese Grand Prix 2022'
 'Monaco Grand Prix 2023' 'Belgian Grand Prix 2023'
 'Dutch Grand Prix 2023' 'Canadian Grand Prix 2024'
 'Spanish Grand Prix 2024' 'British Grand Prix 2024'
 'São Paulo Grand Prix 2024' 'Australian Grand Prix 2025'
 'Miami Grand Prix 2025' 'British Grand Prix 2025']


### Race-Level Summary (Rainy Laps)

In [23]:
rain_race_results_list = []

for race_id in df_rain['RaceID'].unique():
    race_data = df_rain[df_rain['RaceID'] == race_id]

    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()

    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()

        if len(driver_race) == 0:
            continue

        event_name = race_data['EventName'].iloc[0]
        year = race_data['Year'].iloc[0]
        location = race_data['Location'].iloc[0]

        valid_positions = driver_race['Position'].dropna()
        if len(valid_positions) == 0:
            continue

        first_lap = driver_race[driver_race['LapNumber'] == driver_race['LapNumber'].min()]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan

        finishing_pos = valid_positions.iloc[-1]
        best_pos = valid_positions.min()
        worst_pos = valid_positions.max()
        avg_pos = valid_positions.mean()
        pos_std = valid_positions.std()

        positions_gained = (grid_pos - finishing_pos) if pd.notna(grid_pos) else np.nan
        laps_led = len(driver_race[driver_race['Position'] == 1.0])
        pit_stops = len(driver_race[driver_race['IsPitLap'] == True])  # always 0 after filter, kept for schema parity

        max_laps_any_driver = race_data['LapNumber'].max()
        driver_max_laps = driver_race['LapNumber'].max()
        dnf = driver_max_laps < (max_laps_any_driver - 2)

        rain_race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'EventName': event_name,
            'Year': year,
            'Location': location,
            'GridPosition': grid_pos,
            'FinishingPosition': finishing_pos,
            'BestPosition': best_pos,
            'WorstPosition': worst_pos,
            'AvgPosition': avg_pos,
            'PositionStdDev': pos_std,
            'PositionsGained': positions_gained,
            'LapsLed': laps_led,
            'RainyLapsCompleted': int(driver_max_laps),
            'DNF': dnf
        })

rain_race_results_df = pd.DataFrame(rain_race_results_list)
print(f"rain_race_results_df: {rain_race_results_df.shape}")
print(f"\nFirst race sample:")
print(rain_race_results_df.head(10))

rain_race_results_df: (404, 16)

First race sample:
  Driver  DriverNumber             Team           EventName  Year   Location  \
0    PER            11     Aston Martin  Spanish Grand Prix  2018  Barcelona   
1    ALO            14          McLaren  Spanish Grand Prix  2018  Barcelona   
2    LEC            16      Kick Sauber  Spanish Grand Prix  2018  Barcelona   
3    STR            18         Williams  Spanish Grand Prix  2018  Barcelona   
4    VAN             2          McLaren  Spanish Grand Prix  2018  Barcelona   
5    MAG            20             Haas  Spanish Grand Prix  2018  Barcelona   
6    HAR            28     Racing Bulls  Spanish Grand Prix  2018  Barcelona   
7    RIC             3  Red Bull Racing  Spanish Grand Prix  2018  Barcelona   
8    OCO            31     Aston Martin  Spanish Grand Prix  2018  Barcelona   
9    VER            33  Red Bull Racing  Spanish Grand Prix  2018  Barcelona   

   GridPosition  FinishingPosition  BestPosition  WorstPosition  Av

### Driver-Level Statistics (Rainy Laps)

In [24]:
rain_driver_stats_list = []

for driver in rain_race_results_df['Driver'].unique():
    driver_races = rain_race_results_df[rain_race_results_df['Driver'] == driver]

    driver_num = driver_races['DriverNumber'].iloc[0]
    team = driver_races['Team'].iloc[0]

    races_entered = len(driver_races)
    races_completed = len(driver_races[driver_races['DNF'] == False])
    dnf_count = races_entered - races_completed

    avg_grid_pos = driver_races['GridPosition'].mean()
    avg_finish_pos = driver_races['FinishingPosition'].mean()
    avg_positions_gained = driver_races['PositionsGained'].mean()
    races_with_gains = len(driver_races[driver_races['PositionsGained'] > 0])
    races_with_losses = len(driver_races[driver_races['PositionsGained'] < 0])

    podiums = len(driver_races[driver_races['FinishingPosition'] <= 3])
    poles = len(driver_races[driver_races['GridPosition'] == 1.0])
    wins = len(driver_races[driver_races['FinishingPosition'] == 1.0])
    total_laps_led = driver_races['LapsLed'].sum()
    avg_position_std = driver_races['PositionStdDev'].mean()

    rain_driver_stats_list.append({
        'Driver': driver,
        'DriverNumber': driver_num,
        'Team': team,
        'RainyRacesEntered': races_entered,
        'RainyRacesCompleted': races_completed,
        'DNFCount': dnf_count,
        'DNFRate': dnf_count / races_entered if races_entered > 0 else 0,
        'AvgGridPosition': avg_grid_pos,
        'AvgFinishPosition': avg_finish_pos,
        'AvgPositionsGained': avg_positions_gained,
        'RacesWithGains': races_with_gains,
        'RacesWithLosses': races_with_losses,
        'Podiums': podiums,
        'Poles': poles,
        'Wins': wins,
        'TotalLapsLed': total_laps_led,
        'AvgPositionConsistency': avg_position_std
    })

rain_driver_stats_df = pd.DataFrame(rain_driver_stats_list).sort_values('RainyRacesEntered', ascending=False)
print(f"rain_driver_stats_df: {rain_driver_stats_df.shape}")
print(f"\nTop 10 drivers by rainy races entered:")
print(rain_driver_stats_df.head(10))

rain_driver_stats_df: (40, 17)

Top 10 drivers by rainy races entered:
   Driver  DriverNumber             Team  RainyRacesEntered  \
2     LEC            16      Kick Sauber                 22   
9     VER            33  Red Bull Racing                 21   
3     STR            18         Williams                 20   
11    HAM            44         Mercedes                 20   
8     OCO            31     Aston Martin                 20   
0     PER            11     Aston Martin                 19   
1     ALO            14          McLaren                 19   
15    BOT            77         Mercedes                 19   
13    SAI            55           Alpine                 19   
17    GAS            10     Racing Bulls                 19   

    RainyRacesCompleted  DNFCount   DNFRate  AvgGridPosition  \
2                    17         5  0.227273         7.545455   
9                    20         1  0.047619         4.047619   
3                    17         3  0.150000

In [25]:
print("Top 10 drivers by average positions gained in rain:")
print(rain_driver_stats_df.nlargest(10, 'AvgPositionsGained')[['Driver', 'Team', 'AvgPositionsGained', 'RainyRacesEntered']])

print("\nMost successful drivers in wet conditions (by wins):")
print(rain_driver_stats_df.nlargest(10, 'Wins')[['Driver', 'Team', 'Wins', 'Podiums', 'Poles']])

Top 10 drivers by average positions gained in rain:
   Driver             Team  AvgPositionsGained  RainyRacesEntered
20    KVY     Racing Bulls            8.000000                  2
24    KUB         Williams            4.000000                  2
12    VET          Ferrari            3.181818                 11
29    LAT         Williams            2.250000                  4
34    LAW     Racing Bulls            2.000000                  4
3     STR         Williams            1.800000                 20
9     VER  Red Bull Racing            1.476190                 21
22    NOR          McLaren            1.470588                 17
21    ALB     Racing Bulls            1.416667                 12
15    BOT         Mercedes            1.210526                 19

Most successful drivers in wet conditions (by wins):
   Driver             Team  Wins  Podiums  Poles
9     VER  Red Bull Racing    11       17      6
11    HAM         Mercedes     5        9      5
7     RIC  Red Bull R

### Position Change Analysis (Rainy Laps)

In [26]:
rain_position_changes_list = []

for race_id in df_rain['RaceID'].unique():
    race_data = df_rain[df_rain['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    location = race_data['Location'].iloc[0]

    for driver in race_data['Driver'].unique():
        driver_data = race_data[race_data['Driver'] == driver].sort_values('LapNumber')
        driver_data = driver_data[driver_data['Position'].notna()]

        if len(driver_data) < 2:
            continue

        team = driver_data['Team'].iloc[0]
        driver_num = driver_data['DriverNumber'].iloc[0]
        positions = driver_data['Position'].values
        lap_numbers = driver_data['LapNumber'].values

        for i in range(1, len(positions)):
            position_change = positions[i-1] - positions[i]
            rain_position_changes_list.append({
                'Driver': driver,
                'DriverNumber': driver_num,
                'Team': team,
                'EventName': event_name,
                'Year': year,
                'Location': location,
                'RaceID': race_id,
                'FromLap': int(lap_numbers[i-1]),
                'ToLap': int(lap_numbers[i]),
                'PositionBefore': positions[i-1],
                'PositionAfter': positions[i],
                'PositionChange': position_change
            })

rain_position_changes_df = pd.DataFrame(rain_position_changes_list)
print(f"rain_position_changes_df: {rain_position_changes_df.shape}")
print(f"\nFirst 10 rainy position changes:")
print(rain_position_changes_df.head(10))

rain_position_changes_df: (5121, 12)

First 10 rainy position changes:
  Driver  DriverNumber          Team           EventName  Year   Location  \
0    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
1    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
2    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
3    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
4    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
5    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
6    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
7    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
8    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   
9    PER            11  Aston Martin  Spanish Grand Prix  2018  Barcelona   

                    RaceID  FromLap  ToLap  PositionBefore  PositionAfter  \
0  S

In [27]:
print("Most dramatic position gains in a single rainy lap:")
print(rain_position_changes_df.nlargest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print("\nMost dramatic position losses in a single rainy lap:")
print(rain_position_changes_df.nsmallest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print(f"\nAverage position change per rainy lap: {rain_position_changes_df['PositionChange'].mean():.3f}")
print(f"Median position change per rainy lap: {rain_position_changes_df['PositionChange'].median():.3f}")

Most dramatic position gains in a single rainy lap:
     Driver                  EventName  Year  FromLap  PositionChange
2231    NOR          German Grand Prix  2019        1            10.0
2548    GAS  Emilia Romagna Grand Prix  2021       16            10.0
2027    STR          German Grand Prix  2019        1             8.0
3849    PIA           Dutch Grand Prix  2023        6             8.0
2049    STR          German Grand Prix  2019       46             7.0
2730    BOT         Russian Grand Prix  2021       49             7.0
2732    RAI         Russian Grand Prix  2021       49             7.0
3450    MSC        Japanese Grand Prix  2022        7             7.0
3762    STR           Dutch Grand Prix  2023       24             7.0
3817    LAW           Dutch Grand Prix  2023        3             7.0

Most dramatic position losses in a single rainy lap:
     Driver            EventName  Year  FromLap  PositionChange
3837    RUS     Dutch Grand Prix  2023        3           -1

---
## Rain vs. Dry Comparison

Comparing key driver metrics between rainy and dry conditions. The dry stats are derived from the overall `driver_stats_df` (all races) minus the rainy contributions approximated via `rain_driver_stats_df`. For a clean split, dry metrics are computed directly from laps where `Rainfall == False`.

In [32]:
# Build dry race-level results using same clean-lap logic
df_dry = df[
    (df['Rainfall'] == False)
    & (df['TrackStatus'] == 1)
    & (df['IsPitLap'] == False)
    & (df['IsTerminalLap'] == False)
].copy()
df_dry['RaceID'] = df_dry['EventName'] + ' ' + df_dry['Year'].astype(str)

dry_race_results_list = []

for race_id in df_dry['RaceID'].unique():
    race_data = df_dry[df_dry['RaceID'] == race_id]
    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()

    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()
        if len(driver_race) == 0:
            continue

        valid_positions = driver_race['Position'].dropna()
        if len(valid_positions) == 0:
            continue

        first_lap = driver_race[driver_race['LapNumber'] == driver_race['LapNumber'].min()]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan
        finishing_pos = valid_positions.iloc[-1]
        positions_gained = (grid_pos - finishing_pos) if pd.notna(grid_pos) else np.nan

        max_laps_any = race_data['LapNumber'].max()
        dnf = driver_race['LapNumber'].max() < (max_laps_any - 2)

        dry_race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'FinishingPosition': finishing_pos,
            'GridPosition': grid_pos,
            'PositionsGained': positions_gained,
            'LapsLed': len(driver_race[driver_race['Position'] == 1.0]),
            'DNF': dnf
        })

dry_race_results_df = pd.DataFrame(dry_race_results_list)
print(f"dry_race_results_df: {dry_race_results_df.shape}")
dry_race_results_df.head()

dry_race_results_df: (3289, 8)


,Driver,DriverNumber,Team,FinishingPosition,GridPosition,PositionsGained,LapsLed,DNF
0,GAS,10,Racing Bulls,16.0,17.0,1.0,0,True
1,PER,11,Aston Martin,11.0,12.0,1.0,0,False
2,ALO,14,McLaren,5.0,10.0,5.0,0,False
3,LEC,16,Kick Sauber,13.0,18.0,5.0,0,False
4,STR,18,Williams,14.0,14.0,0.0,0,False


In [39]:
# Aggregate dry stats per driver
dry_driver_stats_df = (
    dry_race_results_df.groupby(['Driver'])
    .agg(
        DryRacesEntered=('FinishingPosition', 'count'),
        DryAvgFinishPosition=('FinishingPosition', 'mean'),
        DryAvgPositionsGained=('PositionsGained', 'mean'),
        DryDNFCount=('DNF', 'sum'),
        DryWins=('FinishingPosition', lambda x: (x == 1).sum()),
        DryPodiums=('FinishingPosition', lambda x: (x <= 3).sum()),
        DryTotalLapsLed=('LapsLed', 'sum')
    )
    .reset_index()
)
dry_driver_stats_df['DryDNFRate'] = dry_driver_stats_df['DryDNFCount'] / dry_driver_stats_df['DryRacesEntered']
dry_driver_stats_df.head()
# Aggregate rain stats per driver (matching columns)
rain_agg = (
    rain_race_results_df.groupby('Driver')
    .agg(
        RainRacesEntered=('FinishingPosition', 'count'),
        RainAvgFinishPosition=('FinishingPosition', 'mean'),
        RainAvgPositionsGained=('PositionsGained', 'mean'),
        RainDNFCount=('DNF', 'sum'),
        RainWins=('FinishingPosition', lambda x: (x == 1).sum()),
        RainPodiums=('FinishingPosition', lambda x: (x <= 3).sum()),
        RainTotalLapsLed=('LapsLed', 'sum')
    )
    .reset_index()
)
rain_agg['RainDNFRate'] = rain_agg['RainDNFCount'] / rain_agg['RainRacesEntered']

# Merge rain and dry
comparison_df = rain_agg.merge(dry_driver_stats_df, on='Driver', how='inner')

# Key deltas
comparison_df['FinishPosDelta'] = comparison_df['RainAvgFinishPosition'] - comparison_df['DryAvgFinishPosition']
comparison_df['PositionsGainedDelta'] = comparison_df['RainAvgPositionsGained'] - comparison_df['DryAvgPositionsGained']
comparison_df['DNFRateDelta'] = comparison_df['RainDNFRate'] - comparison_df['DryDNFRate']

print(f"comparison_df: {comparison_df.shape}")
print(comparison_df[['Driver', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta',
                       'RainDNFRate', 'DryDNFRate', 'DNFRateDelta']].sort_values('FinishPosDelta').head(20))

comparison_df: (40, 20)
   Driver  RainAvgFinishPosition  DryAvgFinishPosition  FinishPosDelta  \
17    KVY               5.000000             10.947368       -5.947368   
18    LAT              13.250000             15.706897       -2.456897   
16    KUB              15.000000             17.434783       -2.434783   
30    RUS               7.000000              8.946667       -1.946667   
11    GRO              11.200000             13.134615       -1.934615   
38    VET               6.000000              7.887755       -1.887755   
28    RAI               8.500000             10.246753       -1.746753   
7     DEV              13.000000             14.636364       -1.636364   
9     GAS               9.368421             10.728395       -1.359974   
6     COL              14.000000             15.250000       -1.250000   
25    OCO               9.250000             10.397163       -1.147163   
35    TSU              11.866667             12.707547       -0.840881   
22    MAZ     

In [41]:
# Drivers who finish better in rain than dry (negative FinishPosDelta = better position in rain)
print("Drivers who perform BETTER in rain (lower avg finish position):")
better_in_rain = comparison_df[comparison_df['FinishPosDelta'] < 0].sort_values('FinishPosDelta')
print(better_in_rain[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

print("\nDrivers who perform WORSE in rain:")
worse_in_rain = comparison_df[comparison_df['FinishPosDelta'] > 0].sort_values('FinishPosDelta', ascending=False)
print(worse_in_rain[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

Drivers who perform BETTER in rain (lower avg finish position):
Driver  RainRacesEntered  RainAvgFinishPosition  DryAvgFinishPosition  FinishPosDelta
   KVY                 2               5.000000             10.947368       -5.947368
   LAT                 4              13.250000             15.706897       -2.456897
   KUB                 2              15.000000             17.434783       -2.434783
   RUS                18               7.000000              8.946667       -1.946667
   GRO                 5              11.200000             13.134615       -1.934615
   VET                11               6.000000              7.887755       -1.887755
   RAI                 8               8.500000             10.246753       -1.746753
   DEV                 1              13.000000             14.636364       -1.636364
   GAS                19               9.368421             10.728395       -1.359974
   COL                 1              14.000000             15.250000       

In [43]:
# DNF rate comparison
print("DNF rate comparison — Rain vs Dry (sorted by rain DNF rate):")
print(
    comparison_df[['Driver', 'RainRacesEntered', 'RainDNFRate', 'DryDNFRate', 'DNFRateDelta']]
    .sort_values('RainDNFRate', ascending=False)
    .round(3)
    .to_string(index=False)
)

print("\nPositions gained comparison — Rain vs Dry:")
print(
    comparison_df[['Driver', 'RainAvgPositionsGained', 'DryAvgPositionsGained', 'PositionsGainedDelta']]
    .sort_values('PositionsGainedDelta', ascending=False)
    .round(3)
    .to_string(index=False)
)

DNF rate comparison — Rain vs Dry (sorted by rain DNF rate):
Driver  RainRacesEntered  RainDNFRate  DryDNFRate  DNFRateDelta
   COL                 1        1.000       0.042         0.958
   BOR                 2        0.500       0.095         0.405
   MSC                 5        0.400       0.195         0.205
   MAZ                 3        0.333       0.389        -0.056
   BEA                 3        0.333       0.111         0.222
   SIR                 3        0.333       0.158         0.175
   SAR                 6        0.333       0.194         0.139
   ZHO                10        0.300       0.164         0.136
   VET                11        0.273       0.092         0.181
   GAS                19        0.263       0.099         0.164
   VAN                 4        0.250       0.150         0.100
   OCO                20        0.250       0.106         0.144
   LEC                22        0.227       0.087         0.140
   BOT                19        0.211      

---
## Visualisations — Best Driver in Wet vs Normal Conditions

All charts use `comparison_df`, which requires at least one rainy race. Drivers are filtered to those with **≥ 5 rainy races** for statistical reliability.

Charts:
1. Grouped bar — average finish position (rain vs dry)
2. Scatter — rain vs dry finish position with a diagonal reference line
3. Heatmap — multi-metric driver comparison across 6 normalised metrics
4. Bar — finish position delta (who improves most in rain)
5. Grouped bar — average positions gained per lap (rain vs dry)

In [ ]:
import altair as alt

# Filter to drivers with enough rainy race data
MIN_RAIN_RACES = 5
comp = comparison_df[comparison_df['RainRacesEntered'] >= MIN_RAIN_RACES].copy()
comp = comp.sort_values('RainAvgFinishPosition')
print(f"Drivers with >= {MIN_RAIN_RACES} rainy races: {len(comp)}")
print(comp[['Driver', 'RainRacesEntered', 'RainAvgFinishPosition', 'DryAvgFinishPosition', 'FinishPosDelta']].to_string(index=False))

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Drivers with >= 5 rainy races: 25
Driver  RainRacesEntered  RainAvgFinishPosition  DryAvgFinishPosition  FinishPosDelta
   VER                21               2.571429              2.780488       -0.209059
   HAM                20               4.300000              4.173653        0.126347
   PIA                 9               5.222222              5.867647       -0.645425
   VET                11               6.000000              7.887755       -1.887755
   NOR                17               6.294118              6.378378       -0.084261
   RUS                18               7.000000              8.946667       -1.946667
   SAI                19               7.421053              7.113208        0.307845
   LEC                22               8.090909              5.757764        2.333

#### Chart 1 — Average Finish Position: Rain vs Dry

Lower position = better result. Bars sorted by rainy finish position.

In [46]:
# Reshape to long format for grouped bar
finish_long = pd.melt(
    comp,
    id_vars=['Driver'],
    value_vars=['RainAvgFinishPosition', 'DryAvgFinishPosition'],
    var_name='Condition',
    value_name='AvgFinishPosition'
)
finish_long['Condition'] = finish_long['Condition'].map({
    'RainAvgFinishPosition': 'Rain',
    'DryAvgFinishPosition': 'Dry'
})

driver_order = comp['Driver'].tolist()

chart1 = alt.Chart(finish_long).mark_bar().encode(
    x=alt.X('Driver:N', sort=driver_order, title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('AvgFinishPosition:Q', title='Avg Finish Position (lower = better)',
            scale=alt.Scale(domain=[0, 20])),
    color=alt.Color('Condition:N',
                    scale=alt.Scale(domain=['Rain', 'Dry'],
                                    range=['#1f77b4', '#d62728']),
                    legend=alt.Legend(title='Condition')),
    xOffset='Condition:N',
    tooltip=['Driver', 'Condition', alt.Tooltip('AvgFinishPosition:Q', format='.2f')]
).properties(
    title='Average Finish Position — Rain vs Dry',
    width=700,
    height=350
).configure_title(fontSize=15, font='Calibri').configure_axis(labelFontSize=11)

chart1

alt.Chart(...)

#### Chart 2 — Rain vs Dry Finish Position Scatter

Points above the diagonal perform **worse** in rain; points below perform **better** in rain. Coloured by team.

In [48]:
# Diagonal reference line (y = x)
ref_line = alt.Chart(pd.DataFrame({'x': [1, 20], 'y': [1, 20]})).mark_line(
    color='gray', strokeDash=[4, 4], opacity=0.6
).encode(x='x:Q', y='y:Q')

scatter = alt.Chart(comp).mark_circle(size=90, opacity=0.85).encode(
    x=alt.X('DryAvgFinishPosition:Q', title='Dry Avg Finish Position',
            scale=alt.Scale(domain=[1, 18])),
    y=alt.Y('RainAvgFinishPosition:Q', title='Rain Avg Finish Position',
            scale=alt.Scale(domain=[1, 18])),
    tooltip=['Driver',
             alt.Tooltip('RainAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('DryAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('FinishPosDelta:Q', format='.2f', title='Rain Delta')]
)

labels = alt.Chart(comp).mark_text(dy=-10, fontSize=10).encode(
    x='DryAvgFinishPosition:Q',
    y='RainAvgFinishPosition:Q',
    text='Driver:N'
)

chart2 = (ref_line + scatter + labels).properties(
    title='Rain vs Dry Avg Finish Position (below diagonal = better in rain)',
    width=500,
    height=450
).configure_title(fontSize=14, font='Calibri')

chart2

alt.LayerChart(...)

#### Chart 3 — Heatmap: Multi-Metric Driver Comparison

Six metrics normalised 0–1 (where **1 = best**). Colour intensity shows relative strength. Allows direct cross-metric comparison between drivers.

In [54]:
from sklearn.preprocessing import MinMaxScaler

hm_metrics = {
    'Rain Finish Pos':    ('RainAvgFinishPosition', True),   # True = lower is better (invert)
    'Dry Finish Pos':     ('DryAvgFinishPosition',  True),
    'Rain Pos Gained':    ('RainAvgPositionsGained', False),  # higher is better
    'Dry Pos Gained':     ('DryAvgPositionsGained',  False),
    'Rain DNF Rate':      ('RainDNFRate',            True),   # lower is better (invert)
    'Dry DNF Rate':       ('DryDNFRate',             True),
}

hm_rows = []
for label, (col, invert) in hm_metrics.items():
    vals = comp[col].values.astype(float)
    # Normalise 0-1
    mn, mx = vals.min(), vals.max()
    norm = (vals - mn) / (mx - mn) if mx > mn else vals * 0
    if invert:
        norm = 1 - norm
    for driver, score, raw in zip(comp['Driver'], norm, vals):
        hm_rows.append({'Driver': driver, 'Metric': label, 'Score': round(score, 3), 'RawValue': round(raw, 3)})

hm_df = pd.DataFrame(hm_rows)

# Sort drivers by rain finish position (best first)
driver_order_hm = comp.sort_values('RainAvgFinishPosition')['Driver'].tolist()
metric_order = list(hm_metrics.keys())

chart3 = alt.Chart(hm_df).mark_rect().encode(
    x=alt.X('Metric:N', sort=metric_order, title=None,
            axis=alt.Axis(labelAngle=-30, labelFontSize=11)),
    y=alt.Y('Driver:N', sort=driver_order_hm, title='Driver',
            axis=alt.Axis(labelFontSize=11)),
    color=alt.Color('Score:Q',
                    scale=alt.Scale(domain=[0, 1], range=['yellow', 'red']),
                    legend=alt.Legend(title='Score (1=best)')),
    tooltip=['Driver', 'Metric',
             alt.Tooltip('Score:Q', format='.3f'),
             alt.Tooltip('RawValue:Q', format='.3f', title='Raw Value')]
).properties(
    title='Driver Performance Heatmap — Rain vs Dry (normalised, 1 = best)',
    width=840,
    height=420
).configure_title(fontSize=14, font='Calibri')

chart3

alt.Chart(...)

#### Chart 4 — Finish Position Delta (Rain − Dry)

Negative = finishes **higher** (better) in rain. Bars sorted by delta.

In [59]:
delta_sorted = comp.sort_values('FinishPosDelta')

chart4 = alt.Chart(delta_sorted).mark_bar().encode(
    x=alt.X('Driver:N', sort=delta_sorted['Driver'].tolist(), title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('FinishPosDelta:Q',
            title='Finish Position Delta (Rain − Dry)',
            axis=alt.Axis(grid=True)),
    color=alt.condition(
        alt.datum.FinishPosDelta < 0,
        alt.value('#1f77b4'),   # blue = better in rain
        alt.value('#d62728')    # red = worse in rain
    ),
    tooltip=['Driver',
             alt.Tooltip('FinishPosDelta:Q', format='.2f', title='Delta (Rain−Dry)'),
             alt.Tooltip('RainAvgFinishPosition:Q', format='.2f'),
             alt.Tooltip('DryAvgFinishPosition:Q', format='.2f')]
).properties(
    title='Finish Position Delta — Rain vs Dry (blue = better in rain)',
    width=840,
    height=320
).configure_title(fontSize=14, font='Calibri').configure_axis(labelFontSize=11)

chart4

alt.Chart(...)

#### Chart 5 — Average Positions Gained: Rain vs Dry

Higher = gains more positions during a race. Reveals overtaking ability under different conditions.

In [58]:
gains_long = pd.melt(
    comp.sort_values('RainAvgPositionsGained', ascending=False),
    id_vars=['Driver'],
    value_vars=['RainAvgPositionsGained', 'DryAvgPositionsGained'],
    var_name='Condition',
    value_name='AvgPositionsGained'
)
gains_long['Condition'] = gains_long['Condition'].map({
    'RainAvgPositionsGained': 'Rain',
    'DryAvgPositionsGained': 'Dry'
})

gains_order = comp.sort_values('RainAvgPositionsGained', ascending=False)['Driver'].tolist()

chart5 = alt.Chart(gains_long).mark_bar().encode(
    x=alt.X('Driver:N', sort=gains_order, title='Driver',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('AvgPositionsGained:Q', title='Avg Positions Gained per Race'),
    color=alt.Color('Condition:N',
                    scale=alt.Scale(domain=['Rain', 'Dry'],
                                    range=['#1f77b4', '#d62728']),
                    legend=alt.Legend(title='Condition')),
    xOffset='Condition:N',
    tooltip=['Driver', 'Condition', alt.Tooltip('AvgPositionsGained:Q', format='.2f')]
).properties(
    title='Avg Positions Gained per Race — Rain vs Dry',
    width=840,
    height=320
).configure_title(fontSize=14, font='Calibri').configure_axis(labelFontSize=11)

chart5

alt.Chart(...)